In [ ]:
import glob
import os
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms
from ultralytics import SAM, YOLO

# -----------------------------------------------------------------------------
# 1. Device Setup, Class Mapping & Model Loading
# -----------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------------------------------------------------------
# Automatic Weights Checker & Downloader
# -----------------------------------------------------------------------------
# Replace 'YOUR_USERNAME' and 'YOUR_REPO' with your actual GitHub repository details.
WEIGHTS_URLS = {
    "best.pt": "https://github.com/Coffee-Milk-Sugar/MiniProject-Project-50/releases/download/custom_model_weights/best.pt",
    "sam2.1_b.pt": "https://github.com/Coffee-Milk-Sugar/MiniProject-Project-50/releases/download/custom_model_weights/sam2.1_b.pt",
    "resnet18_traffic_sign.pth": "https://github.com/Coffee-Milk-Sugar/MiniProject-Project-50/releases/download/custom_model_weights/resnet18_traffic_sign.pth",
}

for filename, url in WEIGHTS_URLS.items():
    if not os.path.exists(filename):
        print(f"Downloading missing weight file: {filename}...")
        # PyTorch's built-in downloader displays a neat progress bar
        torch.hub.download_url_to_file(url, filename, progress=True)
        print(f"Successfully downloaded {filename}.")



# Target class map setup
CLASS_ID_MAP = {
    "000": "Sign 5",
    "001": "Sign 15",
    "002": "Sign 30",
    "003": "Sign 40",
    "004": "Sign 50",
    "005": "Sign 60",
    "007": "Sign 80",
    "008": "No Proceed Straight or Turn Left",
    "010": "No Proceed Straight",
    "011": "No Turn Left",
    "012": "No Turn Left or Right",
    "013": "No Turn Right",
    "014": "No Overtaking",
    "015": "No U-Turn",
    "016": "No Cars Allowed",
    "017": "No Sound Horn",
    "020": "Proceed Straight or Turn Right",
    "021": "Proceed Straight",
    "022": "Turn Left",
    "023": "Turn Left or Right",
    "024": "Turn Right",
    "026": "Keep Right",
    "027": "Roundabout",
    "028": "Cars Only",
    "029": "Compulsory Sound Horn",
    "030": "Bicycle Lane",
    "031": "U-Turn",
    "032": "Detour Left or Right",
    "033": "Traffic Signals Ahead",
    "034": "Danger Ahead",
    "035": "Pedestrian Crossing Ahead",
    "036": "Bicycle Crossing Ahead",
    "037": "School Zone Ahead",
    "038": "Sharp Curve to the Right",
    "040": "Steep Descent",
    "042": "Slow Down",
    "043": "T-Intersection Ahead",
    "045": "Village Ahead",
    "046": "Continuous Curves Ahead",
    "047": "Railway Crossing Ahead",
    "048": "Roadworks Ahead",
    "049": "Electrical Hazard",
    "050": "Fences Ahead",
    "051": "Accident-Prone Road Sections",
    "052": "Stop Sign",
    "055": "No Entry",
    "056": "Yield Sign",
    "057": "Inspection",
}

# NOTE / ACTION ITEM:
# This derives class index -> label by sorting the *label strings*
# alphabetically. That is only correct if the ResNet-18 was trained with
# class indices assigned in that same order (e.g. via a manually sorted
# list of label names). If training instead used something like
# torchvision.datasets.ImageFolder over directories named by the numeric
# CLASS_ID_MAP keys ("000", "001", ...), the true index order follows the
# sorted *codes*, not the sorted *label names*, and this mapping will
# silently mislabel every prediction (no error, just wrong text on output).
# Before trusting this pipeline's output, verify against whatever
# class_to_idx / idx_to_class mapping (if any) was saved during training.
unique_targets = sorted(list(set(CLASS_ID_MAP.values())))
idx_to_class = {i: target for i, target in enumerate(unique_targets)}

# Model A: YOLO Detection
yolo_model = YOLO("best.pt")

# Model B: SAM 2.1 Segmentation
sam_model = SAM("sam2.1_b.pt")

# Model C: ResNet-18 Classification
num_classes = len(unique_targets)
resnet_model = models.resnet18(weights=None)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, num_classes)

try:
    state_dict = torch.load(
        "resnet18_traffic_sign.pth", map_location=device, weights_only=True
    )
except Exception:
    # Fall back for checkpoints that aren't pure state dicts / older torch versions.
    state_dict = torch.load("resnet18_traffic_sign.pth", map_location=device)

resnet_model.load_state_dict(state_dict)
resnet_model = resnet_model.to(device)
resnet_model.eval()

# ResNet Transform Pipeline
infer_transforms = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# -----------------------------------------------------------------------------
# 2. Path Setup & File Loading
# -----------------------------------------------------------------------------
input_dir = os.path.join(".", "InputTest\\*")
output_shaped_crops_dir = os.path.join(".", "Outputs", "ShapedCrops_SAM2")
output_annotated_dir = os.path.join(".", "Outputs", "Annotated_Detections")

os.makedirs(output_shaped_crops_dir, exist_ok=True)
os.makedirs(output_annotated_dir, exist_ok=True)

img_paths = []
for ext in ("png", "jpg", "jpeg", "ppm"):
    # Case-insensitive match (handles .PNG, .Jpg, etc.) without duplicating
    # matches on case-insensitive filesystems.
    pattern = os.path.join(input_dir, f"*.{ext}")
    for p in glob.glob(pattern):
        img_paths.append(p)
    pattern_upper = os.path.join(input_dir, f"*.{ext.upper()}")
    for p in glob.glob(pattern_upper):
        if p not in img_paths:
            img_paths.append(p)

print(f"Loaded {len(img_paths)} test images.")

window_name = "Traffic Sign Pipeline (Left: Original | Middle: SAM Mask | Right: Segmented + ResNet Class)"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

# Fixed display width every panel is normalized to before a caption is drawn.
# Captions are rendered at this resolution -- not the source image's native
# resolution -- so tiny source images (e.g. small crops) don't force the font
# down to a few pixels tall and then get blurrily upscaled for viewing.
DISPLAY_PANEL_WIDTH = 480
CAPTION_BAR_HEIGHT = 44


def resize_to_display_width(img, target_width=DISPLAY_PANEL_WIDTH):
    """Resize (up or down) to a fixed width, preserving aspect ratio, so every
    panel -- regardless of the source image's original resolution -- is
    normalized to the same pixel density before captions/text are drawn."""
    h, w = img.shape[:2]
    if w == target_width:
        return img.copy()
    scale = target_width / float(w)
    new_h = max(1, int(round(h * scale)))
    # INTER_CUBIC for upscaling (smoother enlargement of small images),
    # INTER_AREA for downscaling (better quality than linear/nearest).
    interp = cv2.INTER_CUBIC if scale > 1.0 else cv2.INTER_AREA
    return cv2.resize(img, (target_width, new_h), interpolation=interp)


def add_caption(img, text, bar_height=CAPTION_BAR_HEIGHT, bg_color=(20, 20, 20), text_color=(255, 255, 255)):
    """Return img with a caption bar appended below it (dark strip + centered
    text), matching the 'Original' / 'Sign 60' style label bar. Assumes img
    has already been normalized to DISPLAY_PANEL_WIDTH via
    resize_to_display_width, so the font renders crisply at a consistent
    size regardless of the source image's original resolution."""
    h, w = img.shape[:2]
    bar = np.full((bar_height, w, 3), bg_color, dtype=np.uint8)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)

    # Shrink font automatically if the caption is wider than the panel.
    while tw > w - 10 and font_scale > 0.3:
        font_scale -= 0.05
        (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)

    text_x = max(5, (w - tw) // 2)
    text_y = (bar_height + th) // 2
    cv2.putText(
        bar, text, (text_x, text_y), font, font_scale, text_color, thickness, cv2.LINE_AA
    )
    return np.vstack([img, bar])

# -----------------------------------------------------------------------------
# 3. Integrated Sequential Processing Pipeline
# -----------------------------------------------------------------------------
for img_path in img_paths:
    base_name = os.path.splitext(os.path.basename(img_path))[0]

    # Step A: YOLO Box Detection
    yolo_results = yolo_model(img_path, conf=0.20, iou=0.5, device=device)

    for result in yolo_results:
        orig_img = result.orig_img.copy()
        orig_h, orig_w = orig_img.shape[:2]

        vis_img = orig_img.copy()  # Middle panel canvas
        combined_sam_mask = np.zeros(
            (orig_h, orig_w), dtype=np.uint8
        )  # Cumulative mask

        # Reset per-image (this used to persist across images via `locals()`
        # and would silently re-annotate the next image with stale boxes).
        resnet_predictions = []

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()

            point_prompts = []
            point_labels = []

            for box in boxes:
                x1, y1, x2, y2 = box
                center_x = (x1 + x2) / 2.0
                center_y = (y1 + y2) / 2.0
                point_prompts.append([[center_x, center_y]])
                point_labels.append([1])

            # Step B: SAM 2.1 Prompted Segmentation
            sam_results = sam_model(
                img_path,
                bboxes=boxes.tolist(),
                points=point_prompts,
                labels=point_labels,
                device=device,
            )

            if sam_results and sam_results[0].masks is not None:
                masks = sam_results[0].masks.data.cpu().numpy()

                for idx, (box, conf, raw_mask) in enumerate(
                    zip(boxes, confs, masks)
                ):
                    x1, y1, x2, y2 = map(int, box)

                    # Binary Mask Generation
                    binary_mask = (raw_mask > 0).astype(np.uint8) * 255
                    if binary_mask.shape[:2] != (orig_h, orig_w):
                        binary_mask = cv2.resize(
                            binary_mask,
                            (orig_w, orig_h),
                            interpolation=cv2.INTER_NEAREST,
                        )

                    # --- 1. MORPHOLOGICAL CLEANUP ---
                    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
                    cleaned_mask = cv2.morphologyEx(
                        binary_mask, cv2.MORPH_CLOSE, kernel, iterations=2
                    )
                    cleaned_mask = cv2.morphologyEx(
                        cleaned_mask, cv2.MORPH_OPEN, kernel, iterations=1
                    )

                    # --- 2. CONNECTED COMPONENT FILTERING ---
                    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
                        cleaned_mask
                    )
                    if num_labels > 1:
                        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
                        binary_mask = np.zeros_like(cleaned_mask)
                        binary_mask[labels == largest_label] = 255
                    else:
                        binary_mask = cleaned_mask

                    # --- 3. GEOMETRIC POLYGON FITTING (CONVEX HULL) ---
                    contours, _ = cv2.findContours(
                        binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                    )
                    if contours:
                        largest_contour = max(contours, key=cv2.contourArea)
                        hull = cv2.convexHull(largest_contour)

                        poly_mask = np.zeros_like(binary_mask)
                        cv2.drawContours(
                            poly_mask, [hull], -1, (255,), thickness=-1
                        )
                        binary_mask = poly_mask

                    combined_sam_mask = cv2.bitwise_or(
                        combined_sam_mask, binary_mask
                    )

                    # --- 4. EXPLICIT 4-CHANNEL BGRA TRANSPARENT CROP EXPORT ---
                    masked_bgr = cv2.bitwise_and(orig_img, orig_img, mask=binary_mask)
                    bgra_img = cv2.cvtColor(masked_bgr, cv2.COLOR_BGR2BGRA)
                    bgra_img[:, :, 3] = binary_mask

                    x1_c, y1_c = max(0, x1), max(0, y1)
                    x2_c, y2_c = min(orig_w, x2), min(orig_h, y2)
                    cropped_shape = bgra_img[y1_c:y2_c, x1_c:x2_c]

                    if cropped_shape.size > 0:
                        out_path = os.path.join(
                            output_shaped_crops_dir,
                            f"{base_name}_sign_{idx}.png",
                        )
                        cv2.imwrite(out_path, cropped_shape)

                    # --- 5. RESNET-18 INFERENCE ON SEGMENTED CROP ---
                    crop_bgr = masked_bgr[y1_c:y2_c, x1_c:x2_c]
                    if crop_bgr.size > 0:
                        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
                        pil_crop = Image.fromarray(crop_rgb)
                        input_tensor = (
                            infer_transforms(pil_crop).unsqueeze(0).to(device)
                        )

                        with torch.no_grad():
                            outputs = resnet_model(input_tensor)
                            probs = torch.softmax(outputs, dim=1)
                            cls_conf, pred_idx = torch.max(probs, 1)
                            pred_class_name = idx_to_class[pred_idx.item()]
                            cls_score = cls_conf.item()
                    else:
                        pred_class_name = "Unknown"
                        cls_score = 0.0

                    resnet_predictions.append(
                        (box, pred_class_name, cls_score)
                    )

                    # Middle Panel Annotations (Bounding Box + Cyan SAM Mask Overlay)
                    mask_colored = np.zeros_like(vis_img)
                    mask_colored[binary_mask == 255] = (255, 255, 0)
                    vis_img = cv2.addWeighted(
                        vis_img, 1.0, mask_colored, 0.4, 0
                    )

                    cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    sam_label = f"SAM 2.1 ({conf:.2f})"
                    (tw, th), _ = cv2.getTextSize(
                        sam_label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1
                    )
                    cv2.rectangle(
                        vis_img, (x1, y1 - th - 6), (x1 + tw, y1), (0, 255, 0), -1
                    )
                    cv2.putText(
                        vis_img,
                        sam_label,
                        (x1, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.45,
                        (0, 0, 0),
                        1,
                        cv2.LINE_AA,
                    )

                # Save Middle Panel Image
                annotated_path = os.path.join(
                    output_annotated_dir, f"{base_name}_annotated.png"
                )
                cv2.imwrite(annotated_path, vis_img)

        # Step C: Construct Right Panel (SAM Segmented + ResNet Class Labeling)
        segmented_full_img = cv2.bitwise_and(
            orig_img, orig_img, mask=combined_sam_mask
        )

        # Annotate Right Panel with ResNet Classifications.
        # (resnet_predictions is freshly reset per image above, so this is
        # empty -- and correctly draws nothing -- for images with no
        # detections/masks, instead of reusing a previous image's boxes.)
        # Only the box outline is drawn on the image itself; the predicted
        # class name is shown in the caption bar below instead of as an
        # in-image label, matching the reference screenshot.
        for (box, pred_class_name, cls_score) in resnet_predictions:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(
                segmented_full_img, (x1, y1), (x2, y2), (0, 255, 255), 2
            )

        # Build the caption text for the right panel: one label per detected
        # sign (with confidence), comma-separated if there's more than one.
        if resnet_predictions:
            right_caption = ", ".join(
                f"{name} ({score * 100:.0f}%)" for (_, name, score) in resnet_predictions
            )
        else:
            right_caption = "No Detection"

        # Step D: Normalize every panel to a fixed display width first (so
        # caption text renders at a consistent, crisp resolution regardless
        # of how small/large the source image was), then add caption bars
        # and stack the 3 panels side-by-side.
        orig_display = resize_to_display_width(orig_img)
        vis_display = resize_to_display_width(vis_img)
        seg_display = resize_to_display_width(segmented_full_img)

        orig_captioned = add_caption(orig_display, "Original")
        vis_captioned = add_caption(vis_display, "SAM Mask")
        seg_captioned = add_caption(seg_display, right_caption)

        partition_view = np.hstack([orig_captioned, vis_captioned, seg_captioned])

        # Display Window Navigation
        cv2.imshow(window_name, partition_view)
        print(
            f"Displaying: {base_name} | Press ANY KEY to step forward (ESC to exit)..."
        )

        key = cv2.waitKey(0) & 0xFF
        if key == 27:  # ESC key
            print("Exiting interactive display...")
            cv2.destroyAllWindows()
            raise SystemExit

cv2.destroyAllWindows()
